# Generating games with GAMUT

[GAMUT](http://gamut.stanford.edu/) is a suite of parameterised game generators covering a wide range of game families studied in the game theory literature. Written in Java, GAMUT can generate instances of 35 game classes, including random games, coordination games, covariant games, voting games, and many more.

PyGambit's `generate_gamut` function calls GAMUT as an external subprocess and returns the resulting game as a `Game` object, ready for analysis with PyGambit's full suite of tools. Before running this tutorial, you will need Java and `gamut.jar` installed; see the [catalog documentation](../../catalog.html#catalog-gamut) for full installation instructions.

> **Note:** The cell outputs in this notebook were generated locally. To reproduce them, update the `gamut_jar` argument in each cell to the path of your local `gamut.jar`.

This tutorial covers:
- Generating classic two-player games with no additional parameters
- Generating parameterised random normal-form games
- Exploring how the covariance parameter in `CovariantGame` affects equilibrium structure
- Generating multi-player games
- Controlling payoff normalisation and obtaining integer payoffs


In [ ]:
import pygambit as gbt

## Classic two-player games

Many of GAMUT's game classes correspond directly to well-known games from the game theory literature and require no additional parameters to generate. Here we generate Battle of the Sexes, a canonical 2×2 coordination game.

In Battle of the Sexes, two players must independently decide whether to go to the Opera or a Football match. Both prefer to attend the same event, but player 1 prefers the Opera while player 2 prefers Football. The game has two pure-strategy Nash equilibria — both go to the Opera, or both go to Football — and one mixed-strategy equilibrium.

We use the `normalize`, `min_payoff`, and `max_payoff` parameters to rescale payoffs to the range [0, 100], making them easier to read:


In [2]:
g_bos = gbt.catalog.generate_gamut(
    "BattleOfTheSexes",
    params={"normalize": True, "min_payoff": 0, "max_payoff": 100},
    gamut_jar="~/Downloads/gamut.jar",
)
g_bos.title = "Battle of the Sexes"
for i, label in enumerate(["Opera", "Football"]):
    g_bos.players[0].strategies[i].label = label
    g_bos.players[1].strategies[i].label = label
g_bos


Game(title='Battle of the Sexes')

We can inspect the payoff matrices directly using `to_arrays`:


In [3]:
p1, p2 = g_bos.to_arrays(dtype=float)
print("Player 1 payoffs:\n", p1)
print("\nPlayer 2 payoffs:\n", p2)


Player 1 payoffs:
 [[100.0 0.0]
 [0.0 44.27621769809304]]

Player 2 payoffs:
 [[44.27621769809304 0.0]
 [0.0 100.0]]


Now let's compute all Nash equilibria using the linear complementarity method, which is well-suited to two-player games:


In [4]:
result_bos = gbt.nash.lcp_solve(g_bos)
result_bos.equilibria


[[[Rational(1, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1)]],
 [[Rational(1250000000000000, 1803452721226163), Rational(553452721226163, 1803452721226163)], [Rational(553452721226163, 1803452721226163), Rational(1250000000000000, 1803452721226163)]],
 [[Rational(0, 1), Rational(1, 1)], [Rational(0, 1), Rational(1, 1)]]]

As expected, the solver finds three equilibria: two pure-strategy equilibria (both play Opera, or both play Football) and one mixed-strategy equilibrium in which each player randomises between the two options.


## Parameterised random normal-form games

Most GAMUT game classes accept parameters to control the number of players, the number of actions, and the structure of payoffs. These are passed as a dictionary to the `params` argument; each key maps to a GAMUT command-line flag.

`RandomGame` is the most general generator: payoffs are drawn independently and uniformly at random from a fixed range. It serves as a useful baseline for testing algorithms. The `players` parameter controls the number of players, and `actions` controls the number of actions per player. When `actions` is a list, each element gives the action count for the corresponding player, allowing asymmetric games:


In [6]:
g_rand = gbt.catalog.generate_gamut(
    "RandomGame",
    params={"players": 2, "actions": [3, 4]},
    gamut_jar="~/Downloads/gamut.jar",
)
g_rand.title = "Random Game (2 players, 3x4)"
g_rand


Game(title='Random Game (2 players, 3x4)')

In [7]:
p1, p2 = g_rand.to_arrays(dtype=float)
print("Player 1 payoffs:\n", p1)
print("\nPlayer 2 payoffs:\n", p2)


Player 1 payoffs:
 [[-24.922576315638636 31.29134905292773 15.412579872383247
  -18.498230186173046]
 [-15.604513815064706 -72.86583393575657 -60.67300378440537
  38.1610749100424]
 [-21.700394801679934 -10.471762471204386 83.02732785577425
  27.372274400224896]]

Player 2 payoffs:
 [[-46.46932714427341 55.82122243305702 65.14942284746837
  49.38832148801541]
 [21.639090280011786 98.13534287102638 -82.94407428214976
  -27.91660410384405]
 [95.96570634744992 -51.882168608786586 48.053199742230674
  -96.1233605871024]]


In [8]:
gbt.nash.lcp_solve(g_rand).equilibria[0]


[[Rational(95637629221277359420347717551372, 249827048381960319980942015377107), Rational(38671994530917040822304290079883, 166551365587973546653961343584738), Rational(192362854728614798654275725411821, 499654096763920639961884030754214)], [Rational(2555033098519244082676003505069118, 2787217584178145372304577207470169), Rational(218800002201863770017740795011731, 2787217584178145372304577207470169), Rational(13384483457037519610832907389320, 2787217584178145372304577207470169), Rational(0, 1)]]

## Covariant games

`CovariantGame` generates two-player games in which the degree of alignment between players' interests is controlled by a covariance parameter `r`.

- When `r = 1` the game is a common-payoff game: both players receive identical payoffs.
- When `r = 0` payoffs are independent — equivalent to `RandomGame`.
- As `r` approaches `-1` (the minimum for a two-player game) the game approaches a zero-sum game.

Let's compare an equilibrium under high positive covariance (nearly a coordination game) with one under negative covariance (a more adversarial setting):


In [9]:
g_cov_pos = gbt.catalog.generate_gamut(
    "CovariantGame",
    params={"players": 2, "actions": [3, 3], "r": 0.8},
    gamut_jar="~/Downloads/gamut.jar",
)
g_cov_pos.title = "Covariant Game (r=0.8)"
eqm_pos = gbt.nash.lcp_solve(g_cov_pos).equilibria[0]
eqm_pos


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)]]

In [10]:
g_cov_neg = gbt.catalog.generate_gamut(
    "CovariantGame",
    params={"players": 2, "actions": [3, 3], "r": -0.5},
    gamut_jar="~/Downloads/gamut.jar",
)
g_cov_neg.title = "Covariant Game (r=-0.5)"
eqm_neg = gbt.nash.lcp_solve(g_cov_neg).equilibria[0]
eqm_neg


[[Rational(0, 1), Rational(0, 1), Rational(1, 1)], [Rational(0, 1), Rational(1, 1), Rational(0, 1)]]

With high positive covariance (`r = 0.8`) the game is close to a coordination game, so an equilibrium in pure or near-pure strategies is typical. With negative covariance (`r = -0.5`) the game is more adversarial, making genuinely mixed equilibria more likely.


## Multi-player games

GAMUT includes several n-player game families. Here we use `MajorityVoting`, a model of a committee in which each player votes for one of several candidates. The candidate who receives the most votes wins, and each player has a privately known preferred candidate.

For games with more than two players, `gbt.nash.support_enumeration_solve` finds Nash equilibria by exhaustive enumeration over strategy supports:


In [11]:
g_mv = gbt.catalog.generate_gamut(
    "MajorityVoting",
    params={"players": 3, "actions": 3},
    gamut_jar="~/Downloads/gamut.jar",
)
g_mv.title = "Majority Voting (3 players, 3 candidates)"
g_mv


Game(title='Majority Voting (3 players, 3 candidates)')

In [12]:
result_mv = gbt.nash.support_enumeration_solve(g_mv)
print(f"Found {len(result_mv.equilibria)} equilibria")
result_mv.equilibria[0]


AttributeError: module 'pygambit.nash' has no attribute 'support_enumeration_solve'

## Payoff normalisation and integer payoffs

By default, GAMUT draws payoffs uniformly from the range [-100, 100]. Four global parameters let you rescale and discretise payoffs:

| Parameter | Type | Effect |
|---|---|---|
| `normalize` | boolean flag | Enable rescaling to [`min_payoff`, `max_payoff`] |
| `min_payoff` | number | Lower bound after normalisation |
| `max_payoff` | number | Upper bound after normalisation |
| `int_payoffs` | boolean flag | Round all payoffs to integers |

Boolean flags are passed with value `True`; the flag is then included on the GAMUT command line without a value token (e.g. `{"normalize": True}` becomes `-normalize`, `{"int_payoffs": True}` becomes `-int_payoffs`).

Integer payoffs are useful when working with Gambit's exact rational arithmetic:


In [13]:
g_int = gbt.catalog.generate_gamut(
    "RandomGame",
    params={
        "players": 2,
        "actions": 3,
        "normalize": True,
        "min_payoff": 0,
        "max_payoff": 10,
        "int_payoffs": True,
    },
    gamut_jar="~/Downloads/gamut.jar",
)
g_int.title = "Random Game (integer payoffs, 0-10)"
p1, p2 = g_int.to_arrays()
print("Player 1 payoffs:\n", p1)
print("\nPlayer 2 payoffs:\n", p2)


Player 1 payoffs:
 [[Rational(22282, 1) Rational(58087, 1) Rational(52134, 1)]
 [Rational(9437, 1) Rational(0, 1) Rational(56473, 1)]
 [Rational(84052, 1) Rational(22670, 1) Rational(71012, 1)]]

Player 2 payoffs:
 [[Rational(37340, 1) Rational(100000, 1) Rational(34593, 1)]
 [Rational(61289, 1) Rational(60433, 1) Rational(44482, 1)]
 [Rational(41379, 1) Rational(93780, 1) Rational(48727, 1)]]
